# 14 | Harvest value: from quoted price to conditional cash

**One family, fixed policy algorithm, bounded development test.** Notebook 13
reconciled the rejected WATER-deferral loss; it did not identify a new winning
policy. Here ordinary WATER eligibility is retained and only two HARVEST value
terms are replaced. Unknown future opponent trades and shop unlocks are never
supplied to the candidate.

Stages: reference review → local tests → installed-engine checks → 156-state
activation screen → at most one endpoint pair → durable results and Plotly views.
No previous experiment is rerun. No install, cloud write, Git push, or submission.


In [1]:
from pathlib import Path
import json, os, signal, subprocess, sys, time
import pandas as pd
import plotly.io as pio
from IPython.display import display
ROOT = Path.cwd().resolve()
if not (ROOT / 'run_harvest.py').exists():
    ROOT = Path.home() / 'kaggriculture_harvest_value'
if not (ROOT / 'run_harvest.py').exists():
    raise FileNotFoundError('Open the notebook from the extracted package folder.')
sys.path.insert(0, str(ROOT))
from visualize import reference_figures, screen_figures, pilot_figures, dashboard
OUT = ROOT / 'outputs'
pio.renderers.default = 'plotly_mimetype'
figures = []
receipt = Path.home() / 'kaggriculture_manual_resume/state/runtime.json'
PYTHON_BIN = json.loads(receipt.read_text())['executable']
if Path(sys.executable).absolute() != Path(PYTHON_BIN).absolute():
    raise RuntimeError('Select Kaggriculture Manual (verified source), then restart the kernel.')
print('LIVE MANUAL WORKSPACE | existing data; no prior-study reruns')
print('Source folder:', ROOT)
print('Interpreter:', PYTHON_BIN)

LIVE MANUAL WORKSPACE | existing data; no prior-study reruns
Source folder: /home/sagemaker-user/kaggriculture_harvest_value
Interpreter: /home/sagemaker-user/projects/kaggriculture/.venv/bin/python


## 1. What the preceding result actually established
The largest negative accounting component was milk price realization: unchanged
volume does not mean unchanged cash. This is an accounting identity, **not proof
that the proposed harvest-value rule will improve the outcome**. The rejected
WATER-deferral rule is not used in either arm below.

In [2]:
previous = json.loads((ROOT / 'reference/notebook13_report.json').read_text())
display(pd.DataFrame([{
    'control_coins': previous['cash_control'],
    'rejected_deferral_coins': previous['cash_defer'],
    'reconciled_difference': previous['cash_delta_reconciled'],
    'new_policy_calls_in_diagnosis': previous['new_policy_calls'],
    'official_submission_score': previous['official_submission_score']
}]))
ledger = pd.read_csv(ROOT / 'reference/notebook13_cash_decomposition.csv')
display(ledger.loc[ledger.player == 0, ['operation','item','units_control','units_defer','cash_delta']])

,control_coins,rejected_deferral_coins,reconciled_difference,new_policy_calls_in_diagnosis,official_submission_score
0,44840.0,44444.0,-396.0,0,None


,operation,item,units_control,units_defer,cash_delta
0,BUY_PRODUCT,WHEAT,48,48,4.0
1,BUY_SEED,WHEAT,6,7,-10.0
2,HIRE,LABOR,30,30,0.0
3,SELL,EGG,44,44,22.0
4,SELL,FERTILIZER,45,49,213.0
5,SELL,MILK,28,28,-465.0
6,SELL,STRAWBERRY,4,4,-2.0
7,SELL,TOMATO,24,24,-62.0
8,SELL,WHEAT,59,57,-97.0
9,SELL,WOOL,28,28,1.0


In [3]:
for figure in reference_figures(ROOT):
    figure.show()
    figures.append(figure)

## 2. Hypothesis and isolation
For market inventory I, currently held same-product shed stock b, and a harvest
lot q, replace **quote(I) × q** with **R(I,b+q) − R(I,b)**. R follows the official
unit-by-unit one-sided sale curve, including the non-accumulating one-coin floor.

This conditional value excludes travel, decay, rival trades, reserves and future
price changes. Those omissions make endpoint testing essential. Arrival-demand
features are logged only; they are not combined into the intervention.

Only HARVEST priority terms change during days 20–21. The null fork, window
boundary checks and archived control action checks must all pass. Full callback
coverage beyond three hired hands on either farm remains a separate open issue.


In [4]:
protocol = json.loads((ROOT / 'PROTOCOL.json').read_text())
display(pd.Series(protocol, name='registered development design').to_frame())
dictionary = pd.read_csv(ROOT / 'feature_dictionary.csv')
display(dictionary[['feature','scope','role','family']])

,registered development design
milestone,14
hypothesis,Nonlinear marginal revenue after current same-...
source_commit,7194116dfc92a8663139611233b5a221dad431a4
active_days_zero_indexed,"[20, 21]"
source_groups,"[[1601, 0], [1601, 1], [1602, 0]]"
screen_steps,"480..527 inclusive, plus 479,528,695,696 for u..."
pilot_selection,lexicographically first activated seed/seat; n...
screen_cap_seconds,90
pilot_cap_seconds,180
pilot_pairs_max,1


,feature,scope,role,family
0,quantity,task,policy_input_or_derived_value,current_lot_value
1,current_quote,task,diagnostic_only,current_lot_value
2,shed_units_ahead,task,policy_input_or_derived_value,current_lot_value
3,headline_value,task,diagnostic_only,current_lot_value
4,lot_value_without_backlog,task,diagnostic_only,current_lot_value
5,marginal_value_after_shed,task,policy_input_or_derived_value,current_lot_value
6,within_lot_price_impact,task,diagnostic_only,current_lot_value
7,backlog_price_impact,task,diagnostic_only,current_lot_value
8,total_quote_overstatement,task,diagnostic_only,current_lot_value
9,marginal_to_headline_ratio,task,diagnostic_only,current_lot_value


In [5]:
# The runner owns hard caps; file streaming keeps notebook output responsive.
def run_stage(stage):
    caps = {'screen': 90, 'pilot': 180}
    if stage not in caps:
        raise ValueError('Unregistered stage')
    OUT.mkdir(exist_ok=True)
    logpath = OUT / (stage + '_notebook_console.txt')
    process = None
    offset = 0
    started = time.monotonic()
    def show_new():
        nonlocal offset
        with logpath.open('r', errors='replace') as stream:
            stream.seek(offset)
            text = stream.read()
            offset = stream.tell()
        if text:
            print(text, end='', flush=True)
    try:
        with logpath.open('w') as console:
            process = subprocess.Popen(
                [PYTHON_BIN, '-u', str(ROOT / 'run_harvest.py'), stage],
                cwd=ROOT, stdout=console, stderr=subprocess.STDOUT,
                start_new_session=True,
            )
            while process.poll() is None:
                show_new()
                if time.monotonic() - started > caps[stage] + 30:
                    raise TimeoutError('Notebook emergency deadline; save and bundle diagnostics.')
                time.sleep(0.25)
            show_new()
            if process.returncode:
                raise RuntimeError(f'{stage} failed. Stop, save and bundle. See {logpath}.')
    except BaseException:
        if process is not None and process.poll() is None:
            os.killpg(process.pid, signal.SIGTERM)
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                os.killpg(process.pid, signal.SIGKILL)
                process.wait()
        raise
    return json.loads((OUT / stage / 'report.json').read_text())

## 3. Screen: no experiment until mechanics, control parity and activation pass
The 90-second stage verifies 405 marginal-sale and 189 demand-phase cases against
the installed engine, then screens 52 observations per source episode. Validation
and holdout seeds are never loaded. A null fork must return the original actions.


In [6]:
screen = run_stage('screen')
display(pd.DataFrame([{k: screen[k] for k in (
    'status','decision','observations','source_episodes','independent_seed_blocks',
    'action_changes','candidate_callback_max_ms','elapsed_seconds')}]))
mechanics = json.loads((OUT / 'mechanics.json').read_text())
print('Installed-engine component checks:', mechanics['sale_cases'], '+', mechanics['demand_cases'])

........................................................................
----------------------------------------------------------------------
Ran 72 tests in 4.723s

OK
{"utc": "2026-09-12T03:32:34.679878+00:00", "stage": "PREFLIGHT_PASSED", "verified_objects": 10}
{"utc": "2026-09-12T03:32:34.731212+00:00", "stage": "MECHANICS_PASSED", "sale_cases": 405, "demand_cases": 189}
{"utc": "2026-09-12T03:32:36.833037+00:00", "stage": "SCREEN_EPISODE_SAVED", "seed": 1601, "seat": 0, "opponent": "livestock_fertilizer", "arm": "coordinated", "states": 52}
{"utc": "2026-09-12T03:32:38.451320+00:00", "stage": "SCREEN_EPISODE_SAVED", "seed": 1601, "seat": 1, "opponent": "livestock_fertilizer", "arm": "coordinated", "states": 52}
{"utc": "2026-09-12T03:32:38.701529+00:00", "stage": "HEARTBEAT", "name": "screen", "elapsed_seconds": 10.0}
{"utc": "2026-09-12T03:32:40.267720+00:00", "stage": "SCREEN_EPISODE_SAVED", "seed": 1602, "seat": 0, "opponent": "livestock_fertilizer", "arm": "coordinated", "s

,status,decision,observations,source_episodes,independent_seed_blocks,action_changes,candidate_callback_max_ms,elapsed_seconds
0,HARVEST_VALUE_SCREEN_COMPLETE,STOP_NO_ACTION_ACTIVATION,156,3,2,0,33.085842,11.529806


Installed-engine component checks: 405 + 189


In [7]:
for figure in screen_figures(ROOT):
    figure.show()
    figures.append(figure)

## 4. Endpoint pilot: at most one pair
The first activated source in seed/seat order is selected, not the source with the
best reward. Each branch replays the same recorded prefix in the true seeded
environment, then evolves continuously under responsive policies. A shared seed
is not a promise of identical later public randomness. Ordinary WATER behavior is
preserved; the earlier deferral and rejected route change are not applied.

The pilot is skipped automatically after no activation. Any negative own-cash,
margin or local-match endpoint rejects promotion. One favorable development pair
still requires fresh groups and distinct opponents; it does not establish a
leaderboard improvement.


In [8]:
pilot = run_stage('pilot')
print('Decision:', pilot['decision'])
print('Completed pairs:', pilot.get('completed_pairs', 0))
if pilot.get('completed_pairs'):
    results = pd.read_csv(OUT / 'pilot/paired_results.csv')
    display(results[['seed','seat','coins_control','coins_candidate','coins_delta',
                     'coin_margin_delta','local_match_score_delta','same_state_action_changes']])
else:
    print('No simulation pair was launched because the feature did not change actions.')

{"utc": "2026-09-12T03:32:42.003249+00:00", "stage": "PREFLIGHT_PASSED", "verified_objects": 10}
{"utc": "2026-09-12T03:32:42.009937+00:00", "stage": "HARVEST_VALUE_PILOT_COMPLETE", "decision": "STOP_NO_ACTION_ACTIVATION"}
Decision: STOP_NO_ACTION_ACTIVATION
Completed pairs: 0
No simulation pair was launched because the feature did not change actions.


In [9]:
for figure in pilot_figures(ROOT):
    figure.show()
    figures.append(figure)

## 5. Checkpoint and report
Activation is not feature importance. Estimated utility is not banked cash.
Local coins and match fractions are not the Kaggle rating. No intervention is
promoted automatically. All successful outputs stay in this folder; no automatic
S3 upload is performed.


In [10]:
path = dashboard(figures, OUT / 'harvest_value_dashboard.html')
review = {
    'screen_decision': screen['decision'],
    'pilot_decision': pilot['decision'],
    'plots': len(figures),
    'official_submission_score': None,
    'github_updated': False,
    'cloud_resources_modified': False,
    'feature_engineering_complete': False,
    'next_action': 'Save this notebook, bundle results, then stop the AWS application.'
}
(OUT / 'notebook_review.json').write_text(json.dumps(review, indent=2))
print('Dashboard:', path)
print('NOTEBOOK14_COMPLETE')
print(json.dumps(review, indent=2))

Dashboard: /home/sagemaker-user/kaggriculture_harvest_value/outputs/harvest_value_dashboard.html
NOTEBOOK14_COMPLETE
{
  "screen_decision": "STOP_NO_ACTION_ACTIVATION",
  "pilot_decision": "STOP_NO_ACTION_ACTIVATION",
  "plots": 8,
  "official_submission_score": null,
  "github_updated": false,
  "cloud_resources_modified": false,
  "feature_engineering_complete": false,
  "next_action": "Save this notebook, bundle results, then stop the AWS application."
}


**Save with Ctrl+S now.** Then run the terminal bundle command in START_HERE.md.
Return `kaggriculture_harvest_value_results.zip`, not the input ZIP. If any earlier
cell failed, save and bundle anyway—do not repeatedly retry it. Finally stop the
SageMaker application; do not delete the persistent space.
